# Переменные и библиотеки

In [29]:
import cv2
import numpy as np
import serial
import time

OUT_WIDTH = 1000
OUT_HEIGHT = 700
CAMERA_INDEX = 0

# ArUco
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
parameters = cv2.aruco.DetectorParameters()
detector = cv2.aruco.ArucoDetector(aruco_dict, parameters)

# Serial
COM_PORT = "COM16"
BAUD_RATE = 9600
TRIGGER_CMD = "STOP"

# Color
COLOR_DICT = {
    "BLACK":  (65, 65, 65),
    "RED":    (180, 55, 45),
    "ORANGE": (205, 145, 40),
    "YELLOW": (215, 200, 60),
    "GREEN":  (35, 120, 75),
    "BLUE":   (55, 70, 145),
    "PURPLE": (135, 115, 185)
}

# Функции

In [30]:
def get_marker_ordered_crn(pts_4x2, axis_x, axis_y):
    """Упорядочивает 4 угла одного ArUco-маркера относительно осей поля"""
    center = np.mean(pts_4x2, axis=0)
    tl = tr = br = bl = center
    for pt in pts_4x2:
        dx = float(np.dot(pt - center, axis_x))
        dy = float(np.dot(pt - center, axis_y))
        if dx < 0 and dy < 0:
            tl = pt
        elif dx >= 0 and dy < 0:
            tr = pt
        elif dx >= 0 and dy >= 0:
            br = pt
        else:
            bl = pt
    return (tuple(tl), tuple(tr), tuple(br), tuple(bl))

def build_exact_homography(crn_seq):
    """Строит точную перспективную гомографию по всем 12 углам 3 маркеров"""
    it = iter(crn_seq)
    m0 = next(it)
    m1 = next(it)
    m2 = next(it)

    c0 = np.mean(m0, axis = 0)
    c1 = np.mean(m1, axis = 0)
    c2 = np.mean(m2, axis = 0)

    d01 = float(np.linalg.norm(c0 - c1))
    d12 = float(np.linalg.norm(c1 - c2))
    d02 = float(np.linalg.norm(c0 - c2))

    if d01 < 10 or d12 < 10 or d02 < 10:
        return None

    cos0 = abs(float(np.dot(c1 - c0, c2 - c0))) / (d01 * d02)
    cos1 = abs(float(np.dot(c0 - c1, c2 - c1))) / (d01 * d12)
    cos2 = abs(float(np.dot(c0 - c2, c1 - c2))) / (d02 * d12)

    if cos0 <= cos1 and cos0 <= cos2:
        m_br, c_br = m0, c0
        ma, ca, mb, cb = m1, c1, m2, c2
    elif cos1 <= cos0 and cos1 <= cos2:
        m_br, c_br = m1, c1
        ma, ca, mb, cb = m0, c0, m2, c2
    else:
        m_br, c_br = m2, c2
        ma, ca, mb, cb = m0, c0, m1, c1

    v1 = ca - c_br
    v2 = cb - c_br
    cross = float(v1[0] * v2[1] - v1[1] * v2[0])

    if cross < 0:
        m_tr, c_tr = ma, ca
        m_bl, c_bl = mb, cb
    else:
        m_tr, c_tr = mb, cb
        m_bl, c_bl = ma, ca

    axis_x = c_br - c_bl
    axis_y = c_br - c_tr

    src_bl = get_marker_ordered_crn(m_bl, axis_x, axis_y)
    src_br = get_marker_ordered_crn(m_br, axis_x, axis_y)
    src_tr = get_marker_ordered_crn(m_tr, axis_x, axis_y)

    src_pts = np.array(src_bl + src_br + src_tr, dtype = "float32")

    dst_pts = np.array((
        (54.0, 613.0), (119.0, 613.0), (119.0, 678.0), (54.0, 678.0),
        (805.0, 613.0), (870.0, 613.0), (870.0, 678.0), (805.0, 678.0),
        (897.0, 51.0), (962.0, 51.0), (962.0, 116.0), (897.0, 116.0)
    ), dtype = "float32")

    H, _ = cv2.findHomography(src_pts, dst_pts)
    return H

def analyze_cell(hsv_roi):
    """Определяет наличие шашки (1 или 0) и ее цвет по словарю COLOR_DICT"""
    h, w, _ = hsv_roi.shape
    if h == 0 or w == 0:
        return 0, None, None

    center_crop = hsv_roi[int(h * 0.3):int(h * 0.7), int(w * 0.3):int(w * 0.7)]
    
    median_hsv = np.median(center_crop.reshape(-1, 3), axis=0)
    
    sat = float(median_hsv[1])
    val = float(median_hsv[2])

    if sat < 70 and val >= 75:
        return 0, None, None

    mean_hsv_pixel = np.uint8([[median_hsv]])
    mean_rgb = cv2.cvtColor(mean_hsv_pixel, cv2.COLOR_HSV2RGB)[0][0].astype(float)

    best_color_name = None
    best_rgb = (0, 0, 0)
    min_dist = float("inf")

    for name, rgb in COLOR_DICT.items():
        if name == "BLACK" and sat >= 70:
            continue
            
        dist = np.linalg.norm(mean_rgb - np.array(rgb, dtype=float))
        if dist < min_dist:
            min_dist = dist
            best_color_name = name
            best_rgb = rgb

    if min_dist > 85:
        return 0, None, None

    bgr_color = (int(best_rgb[2]), int(best_rgb[1]), int(best_rgb[0]))
    if best_color_name == "BLACK":
        bgr_color = (50, 50, 50)

    return 1, best_color_name, bgr_color

def send_array(ser, array):
  """Отправляет массив на контроллер"""
  for row in array:
    line = f'{row[0]},{row[1]}\n'
    ser.write(line.encode('utf-8'))
    time.sleep(0.005)
  ser.write(b'E\n')  # Конец передачи


# Main

In [31]:
def main():
    # Подключение по Serial
    ser = None
    try:
        ser = serial.Serial(COM_PORT, BAUD_RATE, timeout = 0.01)
        time.sleep(2.0)
        ser.reset_input_buffer()
        print(f"UART connected ({COM_PORT})")
    except serial.SerialException as e:
        print(f"UART ERROR ({COM_PORT}): {e}")
        return None, None, None

    # Подключение к камере
    cap = cv2.VideoCapture(CAMERA_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1480)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 820)
    if not cap.isOpened():
        print("Camera ERROR")
        if ser and ser.is_open:
            ser.close()
        return None, None, None
    print("Window opened")

    # Координаты сетки
    m_left = 0.021
    m_right = 0.964
    m_top = 0.140
    m_bottom = 0.948

    ls_val_mtx = None
    saved_grid = np.zeros((6, 9), dtype = int)
    saved_color = np.full((6, 9), None, dtype = object)

    # Основной цикл
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        crn, ids, _ = detector.detectMarkers(frame)

        if ids is not None and len(ids) >= 3:
            uniq_marks = {}
            flat_ids = ids.flatten()
            for idx in range(len(flat_ids)):
                m_id = int(flat_ids.item(idx))
                if m_id not in uniq_marks and len(uniq_marks) < 3:
                    uniq_marks[m_id] = crn[idx].reshape(4, 2)

            if len(uniq_marks) == 3:
                H = build_exact_homography(uniq_marks.values())
                if H is not None:
                    ls_val_mtx = H

        if ls_val_mtx is not None:
            warped = cv2.warpPerspective(frame, ls_val_mtx, (OUT_WIDTH, OUT_HEIGHT))
            warped = cv2.flip(warped, -1)
            
            hsv_warped = cv2.cvtColor(warped, cv2.COLOR_BGR2HSV)
            display_img = warped.copy()

            x_start = int(OUT_WIDTH * m_left)
            x_end = int(OUT_WIDTH * m_right)
            y_start = int(OUT_HEIGHT * m_top)
            y_end = int(OUT_HEIGHT * m_bottom)

            cell_w = (x_end - x_start) / 10.0
            cell_h = (y_end - y_start) / 6.0

            binary_grid = np.zeros((6, 9), dtype = int)
            color_grid = np.full((6, 9), None, dtype = object)

            for r in range(6):
                for c in range(10):
                    if c == 0:
                        continue

                    x1 = int(x_start + c * cell_w)
                    y1 = int(y_start + r * cell_h)
                    x2 = int(x1 + cell_w)
                    y2 = int(y1 + cell_h)

                    # Рисование сетки
                    cv2.rectangle(display_img, (x1, y1), (x2, y2), (0, 0, 255), 1)

                    cell_hsv = hsv_warped[y1:y2, x1:x2]
                    chck, clr_nm, bgr_color = analyze_cell(cell_hsv)

                    binary_grid[r, c - 1] = chck
                    color_grid[r, c - 1] = clr_nm

                    if chck == 1:
                        center_x = int(x1 + cell_w / 2)
                        center_y = int(y1 + cell_h / 2)
                        cv2.circle(display_img, (center_x, center_y), 16, bgr_color, -1)
                        cv2.circle(display_img, (center_x, center_y), 16, (0, 0, 0), 2)
                        cv2.putText(display_img, clr_nm, (x1 + 8, y2 - 8),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

            saved_grid = binary_grid.copy()
            saved_color = color_grid.copy()

            total_count = int(np.sum(binary_grid))
            cv2.putText(display_img, f"{total_count}", (20, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

            cv2.imshow("Camera", display_img)
        else:
            if ids is not None:
                cv2.aruco.drawDetectedMarkers(frame, crn, ids)
            cv2.imshow("Camera", frame)

        if ser and ser.in_waiting > 0:
            raw_packet = ser.readline().decode("utf-8", errors = "ignore").strip()
            if raw_packet == TRIGGER_CMD:
                break
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Window closed")

    return saved_grid, saved_color, ser

In [32]:
matrix, colors, arduino = main()

print(matrix)
if matrix is not None:    
    data = []
    for r in range(5, -1, -1):
        for c in range(8, -1, -1):
            if matrix[r][c] == 1:
                data.append([5 - r, 8 - c])

    print(data)
    send_array(arduino, data)
    
    time.sleep(0.1)
    arduino.close()
    print(f"Данные отправлены, порт ({COM_PORT}) закрыт")

UART connected (COM16)
Window opened
Window closed
[[1 0 0 0 0 0 0 0 1]
 [0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0]
 [0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0]
 [1 0 0 0 0 0 0 0 0]]
[[0, 8], [1, 2], [2, 6], [3, 1], [4, 5], [5, 0], [5, 8]]
Данные отправлены, порт (COM16) закрыт
